In [ ]:
# ==================================================
# INSTALL REQUIRED LIBRARIES
# ==================================================

!pip install opencv-python

!pip install scikit-learn

!pip install matplotlib

In [ ]:
# ==================================================
# IMPORT LIBRARIES
# ==================================================

import os

import cv2

import numpy as np

import matplotlib.pyplot as plt

from google.colab import drive

from sklearn.decomposition import PCA

from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score

from sklearn.metrics import classification_report

In [ ]:
# ==================================================
# CONNECT GOOGLE DRIVE
# ==================================================

drive.mount('/content/drive')

In [ ]:
# ==================================================
# DATASET PATH
# ==================================================

DATASET_PATH = "/content/drive/MyDrive/FaceDataset"

In [ ]:
# ==================================================
# LOAD HAAR CASCADE
# ==================================================

face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

In [ ]:
# ==================================================
# LOAD IMAGE DATASET
# ==================================================

images = []

labels = []

label_dict = {}

current_label = 0

IMAGE_SIZE = 100

In [ ]:
# ==================================================
# PREPROCESSING DATASET
# ==================================================

for person_name in os.listdir(DATASET_PATH):

    person_path = os.path.join(
        DATASET_PATH,
        person_name
    )

    if not os.path.isdir(person_path):

        continue

    label_dict[current_label] = person_name

    for image_name in os.listdir(person_path):

        image_path = os.path.join(
            person_path,
            image_name
        )

        image = cv2.imread(image_path)

        if image is None:

            continue

        gray = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2GRAY
        )

        faces = face_detector.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30, 30)
        )

        for (x, y, w, h) in faces:

            face = gray[
                y:y+h,
                x:x+w
            ]

            face = cv2.resize(
                face,
                (IMAGE_SIZE, IMAGE_SIZE)
            )

            # CLAHE PREPROCESSING
            clahe = cv2.createCLAHE(
                clipLimit=2.0,
                tileGridSize=(8, 8)
            )

            face = clahe.apply(face)

            images.append(face)

            labels.append(current_label)

            break

    current_label += 1

In [ ]:
# ==================================================
# CONVERT TO NUMPY ARRAY
# ==================================================

X = np.array(images)

y = np.array(labels)

print("Total Images :", len(X))

print("Total Labels :", len(y))

In [ ]:
# ==================================================
# DISPLAY SAMPLE IMAGES
# ==================================================

plt.figure(figsize=(12, 5))

for i in range(5):

    plt.subplot(1, 5, i + 1)

    plt.imshow(
        X[i],
        cmap='gray'
    )

    plt.title(
        label_dict[y[i]]
    )

    plt.axis("off")

plt.show()

In [ ]:
# ==================================================
# FLATTEN IMAGE
# ==================================================

X_flatten = X.reshape(
    X.shape[0],
    -1
)

print(
    "Flatten Shape :",
    X_flatten.shape
)

In [ ]:
# ==================================================
# FLATTEN IMAGE
# ==================================================

X_flatten = X.reshape(
    X.shape[0],
    -1
)

print(
    "Flatten Shape :",
    X_flatten.shape
)

In [ ]:
# ==================================================
# SPLIT TRAIN AND TEST DATA
# ==================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_flatten,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Size :", len(X_train))

print("Test Size  :", len(X_test))

In [ ]:
# ==================================================
# APPLY PCA (EIGENFACES)
# ==================================================

n_components = 50

pca = PCA(
    n_components=n_components,
    whiten=True,
    random_state=42
)

X_train_pca = pca.fit_transform(
    X_train
)

X_test_pca = pca.transform(
    X_test
)

print(
    "Original Dimension :",
    X_train.shape
)

print(
    "Reduced Dimension :",
    X_train_pca.shape
)

In [ ]:
# ==================================================
# DISPLAY EIGENFACES
# ==================================================

eigenfaces = pca.components_.reshape(
    (
        n_components,
        IMAGE_SIZE,
        IMAGE_SIZE
    )
)

plt.figure(figsize=(12, 6))

for i in range(10):

    plt.subplot(2, 5, i + 1)

    plt.imshow(
        eigenfaces[i],
        cmap='gray'
    )

    plt.title(
        "Eigenface " + str(i + 1)
    )

    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
# ==================================================
# TRAIN KNN CLASSIFIER
# ==================================================

knn_model = KNeighborsClassifier(
    n_neighbors=3
)

knn_model.fit(
    X_train_pca,
    y_train
)

In [ ]:
# ==================================================
# PREDICTION
# ==================================================

y_pred = knn_model.predict(
    X_test_pca
)

In [ ]:
# ==================================================
# MODEL EVALUATION
# ==================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

print(
    "Recognition Accuracy :",
    accuracy
)

In [ ]:
# ==================================================
# CLASSIFICATION REPORT
# ==================================================

print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
# ==================================================
# TEST NEW IMAGE
# ==================================================

TEST_IMAGE_PATH = "/content/drive/MyDrive/test.jpg"

image = cv2.imread(
    TEST_IMAGE_PATH
)

gray = cv2.cvtColor(
    image,
    cv2.COLOR_BGR2GRAY
)

faces = face_detector.detectMultiScale(
    gray,
    scaleFactor=1.1,
    minNeighbors=5
)

for (x, y, w, h) in faces:

    face = gray[
        y:y+h,
        x:x+w
    ]

    face = cv2.resize(
        face,
        (IMAGE_SIZE, IMAGE_SIZE)
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    face = clahe.apply(face)

    face_vector = face.reshape(1, -1)

    face_pca = pca.transform(
        face_vector
    )

    prediction = knn_model.predict(
        face_pca
    )

    predicted_name = label_dict[
        prediction[0]
    ]

    cv2.rectangle(
        image,
        (x, y),
        (x+w, y+h),
        (0, 255, 0),
        2
    )

    cv2.putText(
        image,
        predicted_name,
        (x, y-10),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

plt.figure(figsize=(8, 8))

plt.imshow(
    cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )
)

plt.axis("off")

plt.show()